In [ ]:
import bk_tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from bk_tools import BASE_DIR
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Import all necessary libraries for data manipulation, visualization and deep learning.

In [ ]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Prepare the data table using the custom function from bk_tools and display its information.
df = bk_tools.prepare_data_table()
df.info()
df["class"].unique()

In [ ]:
# Prepare the data splitting for the chosen zoom level (200) and display the class distribution in the validation set.
train_df, val_df, test_df, class_to_idx = bk_tools.prepare_data_splitting_class_v2(
    df=df,
    chosen_zoom=200,   # veya 200, 400 gibi belirli bir değer
    temp_size=0.2,
    seed=42
)
print(val_df["class"].value_counts())

In [ ]:
CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

train_df["label"] = train_df["class"].map(class_to_idx)
val_df["label"] = val_df["class"].map(class_to_idx)
test_df["label"] = test_df["class"].map(class_to_idx)

In [ ]:
from tensorflow.keras.applications.convnext import preprocess_input
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)  # PNG kullanıyorsunuz, JPEG değil!
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)  # ImageNet normalizasyonu
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    # Renk ve parlaklık varyasyonları eklemek overfitting'i azaltır
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
])

In [ ]:
def make_dataset(df, training=False):
    image_paths = df["file_path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()

    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

In [ ]:
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras import applications

def build_pretrained_sequential_model(input_shape=(224, 224, 3), num_classes=8):
    
    base_model = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )

    base_model.trainable = True

    model = models.Sequential([
        tf.keras.Input(shape=input_shape),
        layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x)),        
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.LayerNormalization(epsilon=1e-5),
        layers.Dropout(0.4),
        layers.Dense(
            num_classes, 
            activation="softmax", 
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        )
    ])

    return model, base_model

In [ ]:
from bk_tools import BASE_DIR


model, base_model = build_pretrained_sequential_model(
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc")
    ]
)

MODEL_DIR = BASE_DIR/"models"
MODEL_DIR.mkdir(exist_ok=True)


callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15, # Cosine dalgası bazen geç toparlar, sabrı artırdık
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/best_densenet_model.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
    # ReduceLROnPlateau'ya ARTIK GEREK YOK çünkü CosineDecay bu işi baştan planlıyor.
]
model.summary()

In [ ]:
def adjustClassWeights(smooth_factor=0.3): # 0.15'ten 0.3'e çıkar, makası daralt
    y_train = train_df["label"].values.astype(int)
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    
    # Daha yumuşak bir eğri
    smooth_weights = weights ** (1 - smooth_factor)
    return {cls: weight for cls, weight in zip(classes, smooth_weights)}


def get_advanced_metrics():
    return [
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
        
        tf.keras.metrics.SparseCategoricalCrossentropy(name="cross_entropy"),
    ]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100, # Bırak 100'e kadar gitsin, EarlyStopping en iyi yerde kesecek
    class_weight=adjustClassWeights(smooth_factor=0.3), # Yumuşatılmış ağırlıklarımız
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    history_dict = history.history

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["loss"], label="train_loss")
    plt.plot(history_dict["val_loss"], label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["accuracy"], label="train_accuracy")
    plt.plot(history_dict["val_accuracy"], label="val_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    if "top2_acc" in history_dict and "val_top2_acc" in history_dict:
        plt.figure(figsize=(12, 5))
        plt.plot(history_dict["top2_acc"], label="train_top2_acc")
        plt.plot(history_dict["val_top2_acc"], label="val_top2_acc")
        plt.title("Top-2 Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Top-2 Accuracy")
        plt.legend()
        plt.show()

plot_training_history(history)

In [ ]:
test_loss, cross_ent, test_acc, top2 = model.evaluate(test_ds, verbose=1)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")



y_true = test_df["label"].values
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

In [ ]:
from sklearn.metrics import classification_report

CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, class_names, title="Confusion Matrix"):
    plt.figure(figsize=(9, 7))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    threshold = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, format(cm[i, j], "d"),
                horizontalalignment="center",
                color="white" if cm[i, j] > threshold else "black"
            )

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, CLASS_NAMES)